# 외부 기사 과적합 테스트 v2 — KLUE-RoBERTa 이진 분류

**구성**: 총 50건 (낚시성 25 / 정상 25), AI 생성 25건 + 실제 기사 25건  
**모델**: `klue_binary_final.pt`  
**산출물**: Drive 내 `external_test_result_v2.tsv`

### Google Drive에 업로드할 파일
```
내 드라이브/
└── text-mining-2026/
    ├── klue_binary_final.pt   ← 모델 가중치
    └── external_test_v2.py   ← 샘플 데이터 포함 스크립트
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # torch import 전에 설정해야 효과 있음

import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── 경로 설정 ────────────────────────────────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/text-mining-2026'
MODEL_NAME = 'klue/roberta-base'
BIN_PT     = '/content/drive/MyDrive/text-mining-2026/models/pt_files/klue_binary_final.pt'
SCRIPT_PT  = '/content/drive/MyDrive/text-mining-2026/external_test_v2.py'
MAX_LEN    = 512
LABEL      = {0: '정상', 1: '낚시성'}

# external_test_v2.py에서 SAMPLES 데이터 로드
with open(SCRIPT_PT, 'r', encoding='utf-8') as f:
    _code = f.read()

_code = _code.replace(
    'sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8")', ''
)
_preamble = _code.split('if __name__ == "__main__":')[0]

globals()['__file__'] = SCRIPT_PT
exec(_preamble, globals())

# exec()이 스크립트 내 BIN_PT·MAX_LEN을 덮어쓰므로 다시 지정
BIN_PT  = '/content/drive/MyDrive/text-mining-2026/models/pt_files/klue_binary_final.pt'
MAX_LEN = 512

print(f'모델 파일 존재 : {os.path.exists(BIN_PT)}')
print(f'MAX_LEN        : {MAX_LEN}')
print(f'총 샘플        : {len(SAMPLES)}건')
print(f'낚시성         : {sum(1 for s in SAMPLES if s[2]==1)}건')
print(f'정상           : {sum(1 for s in SAMPLES if s[2]==0)}건')
print(f'AI 생성        : {sum(1 for s in SAMPLES if s[3]=="AI생성")}건')
print(f'실제 기사      : {sum(1 for s in SAMPLES if s[3]=="실제기사")}건')

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {DEVICE}')

print('토크나이저 및 모델 로딩 중...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)
state_dict = torch.load(BIN_PT, map_location='cpu')
model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval()
print('로딩 완료!')

In [ ]:
rows = []
for s in SAMPLES:
    title, content, true_label, src_type = s
    enc = tokenizer(
        text=title,
        text_pair=content,
        truncation='only_second',
        max_length=MAX_LEN,
        padding='max_length',
        return_tensors='pt',
    )
    with torch.no_grad():
        # token_type_ids 제외: KLUE-RoBERTa는 type_vocab_size=1이라
        # text_pair 사용 시 tokenizer가 반환하는 token_type_ids=1이 범위 초과됨
        out = model(
            input_ids=enc['input_ids'].to(DEVICE),
            attention_mask=enc['attention_mask'].to(DEVICE),
        )
    prob = F.softmax(out.logits, dim=-1).squeeze().cpu()
    pred = int(torch.argmax(prob))
    rows.append({
        'No'    : len(rows) + 1,
        '출처유형': src_type,
        '제목'   : title,
        '실제'   : LABEL[true_label],
        '예측'   : LABEL[pred],
        '이진확률': f'{max(prob.tolist()):.3f}',
        '정오'   : '✓' if pred == true_label else '✗',
        '_pred'  : pred,
        '_true'  : true_label,
    })

df = pd.DataFrame(rows)
print(f'추론 완료: {len(df)}건')

In [ ]:
# 결과 테이블
display(df.drop(columns=['_pred', '_true']))

# 정확도 집계
total   = len(df)
correct = (df['_pred'] == df['_true']).sum()

def acc(mask):
    sub = df[mask]
    if len(sub) == 0:
        return 0, 0
    c = (sub['_pred'] == sub['_true']).sum()
    return int(c), len(sub)

ai_c,   ai_n   = acc(df['출처유형'] == 'AI생성')
real_c, real_n = acc(df['출처유형'] == '실제기사')
cb_c,   cb_n   = acc(df['_true'] == 1)
nm_c,   nm_n   = acc(df['_true'] == 0)

print('=' * 50)
print(f'  전체     정확도: {correct}/{total} = {correct/total*100:.1f}%')
print(f'  AI 생성  정확도: {ai_c}/{ai_n} = {ai_c/ai_n*100:.1f}%')
print(f'  실제기사 정확도: {real_c}/{real_n} = {real_c/real_n*100:.1f}%')
print(f'  낚시성   정확도: {cb_c}/{cb_n} = {cb_c/cb_n*100:.1f}%')
print(f'  정상     정확도: {nm_c}/{nm_n} = {nm_c/nm_n*100:.1f}%')
print('=' * 50)

In [ ]:
# TSV 저장 → Drive
out_path = os.path.join(DRIVE_BASE, 'external_test_result_v2.tsv')
df.drop(columns=['_pred', '_true']).to_csv(
    out_path, sep='\t', index=False, encoding='utf-8'
)
print(f'TSV 저장 완료 → {out_path}')